# Optuna search for next-event models

This notebook inspects separate hyperparameter studies for the Seq2Event team-stream adaptation and the full-quarter causal decoder. It reads saved reports without training. During a local run it can display progress from `artifacts/event-optuna-v1`; after export it uses the compact example reports.

Each study fits on **2023–2024**, selects configurations and stopping epochs on **2025**, and refits three seeds on **2023–2025** before evaluating **2026**. All six final models must be frozen before test data is loaded. The earlier experiment remains in Notebook 13.

The two formulations have different targets, class weights and coordinate masks. Their raw scores do not establish a cross-task architecture ranking. The 2026 season is an already-observed benchmark.

## 1. Locate the saved study

Find the repository from the notebook's working directory. Prefer exported reports when available. A running local experiment has a plan and checkpoint histories but no completed results summary. Paths printed below are relative to the repository.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next(p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file() and (p / "data/sources.json").is_file())
report = root / "reports/examples/event-optuna-v1"
if not (report / "summary.json").exists():
    report = root / "artifacts/event-optuna-v1"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

plan = read_json(report / "plan.json") if (report / "plan.json").exists() else None
summary = read_json(report / "summary.json") if (report / "summary.json").exists() else None
print("Snapshot (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Source:", report.relative_to(root).as_posix())
print("Status:", summary["status"] if summary else "In progress" if plan else "Not started")
modes = {"team_stream": "Seq2Event team streams", "joint": "Full-quarter causal decoder"}

Snapshot (UTC): 2026-10-26T00:36:50+00:00
Source: artifacts/event-optuna-v1
Status: In progress


## 2. Inspect the frozen protocol and search space

TPE explores each formulation independently. The initial random trials give it evidence before adaptive sampling begins. Pruning and early stopping wait until epoch 15. The loss definition, data split and target population remain fixed throughout each study. Search-space bounds describe possible values, not claims that every combination was evaluated.

In [2]:
if plan:
    display(pd.DataFrame([{"parameter": key, "values": str(value)} for key, value in plan["search_space"].items()]))
    display(pd.DataFrame([{"setting": key, "value": str(value)} for key, value in plan["protocol"].items()]))
else:
    print("Run the documented Optuna command to create the frozen plan.")

,parameter,values
0,width,"[64, 96, 128, 192, 256]"
1,layers,"[1, 4]"
2,heads,"[2, 4, 8]"
3,embedding_dim,"[8, 16, 32]"
4,ff_multiplier,"[2, 4]"
5,dense_dim,"[32, 64, 128]"
6,dropout,"[0.0, 0.4]"
7,learning_rate,"[1e-05, 0.003]"
8,weight_decay,"[0.0, 1e-06, 0.01]"
9,club_embedding_dim,"[4, 8, 16]"


,setting,value
0,trials,60
1,startup_trials,15
2,max_epochs,60
3,patience,10
4,minimum_epochs,15
5,pruning_startup_trials,10
6,pruning_min_trials,5
7,finalist_count,3
8,seeds,"[2026, 2027, 2028]"
9,batches,"{'team_stream': 512, 'joint': 8}"


## 3. Review search progress

Each completed trial contributes a configuration and its best validation objective. Pruned trials count towards the search budget. The latest checkpoint history can be newer than `trials.csv`, which is refreshed when a trial finishes. The tables show validation results only.

In [3]:
searches = {}
for mode, title in modes.items():
    print(title)
    path = report / mode / "trials.csv"
    if path.exists():
        trials = pd.read_csv(path)
        searches[mode] = trials
        display(trials.groupby("state").size().rename("trials").to_frame())
        columns = ["number", "state", "value"] + [c for c in trials if c.startswith("params_")]
        display(trials.loc[trials.state.eq("COMPLETE"), columns].sort_values("value").head(10))
    histories = sorted((report / mode).glob("trial-*/history.json"))
    if histories:
        latest = histories[-1]
        history = read_json(latest)
        print("Latest checkpoint:", latest.parent.name, "epoch", history[-1]["epoch"])
        display(pd.json_normalize(history[-3:]))
    elif not path.exists():
        print("No trial has completed an epoch yet.")

Seq2Event team streams


,trials
state,
COMPLETE,22
PRUNED,13


,number,state,value,params_dense_dim,params_dropout,params_embedding_dim,params_ff_multiplier,params_heads,params_l2_strength,params_layers,params_learning_rate,params_use_l2,params_width
27,27,COMPLETE,0.175475,128,0.045475,32,2,4,0.000029,4,0.000099,True,256
25,25,COMPLETE,0.175585,128,0.059960,32,2,4,0.000027,3,0.000134,True,256
11,11,COMPLETE,0.175605,128,0.130406,32,2,4,0.000023,4,0.000117,True,256
15,15,COMPLETE,0.175652,32,0.190917,32,4,2,0.000003,3,0.000067,True,192
5,5,COMPLETE,0.175701,32,0.135652,32,4,2,0.000004,3,0.000143,True,192
22,22,COMPLETE,0.175730,128,0.112680,32,4,2,0.000004,2,0.000110,True,192
30,30,COMPLETE,0.175749,32,0.003839,32,2,4,0.000030,4,0.000144,True,256
23,23,COMPLETE,0.175850,32,0.091634,32,4,2,0.000037,4,0.000387,True,192
24,24,COMPLETE,0.176154,32,0.167782,32,4,2,0.000004,3,0.000292,True,192
10,10,COMPLETE,0.176160,32,0.246402,32,2,8,NaN,3,0.000137,False,96


Latest checkpoint: trial-035 epoch 29


,epoch,training_batch_objective,seconds,validation.selection_loss,validation.weighted_cross_entropy,validation.location_rmse_norm
0,27,0.177189,229.360263,0.176327,0.014483,0.103911
1,28,0.177019,237.914367,0.177826,0.014492,0.105368
2,29,0.177036,244.351732,0.176725,0.014450,0.104477


Full-quarter causal decoder


,trials
state,
COMPLETE,36
PRUNED,24


,number,state,value,params_club_embedding_dim,params_dense_dim,params_dropout,params_embedding_dim,params_embedding_dim_compact,params_ff_multiplier,params_heads,params_l2_strength,params_layers,params_learning_rate,params_use_l2,params_width
38,38,COMPLETE,5.531118,4,128,0.261275,8.0,NaN,2,4,NaN,4,0.001900,False,64
46,46,COMPLETE,5.535913,4,128,0.278295,8.0,NaN,2,4,NaN,3,0.001766,False,64
56,56,COMPLETE,5.542192,4,128,0.328471,32.0,NaN,2,4,NaN,4,0.001758,False,64
53,53,COMPLETE,5.542463,4,128,0.258496,32.0,NaN,2,4,NaN,4,0.001337,False,64
51,51,COMPLETE,5.543196,4,128,0.217042,32.0,NaN,2,4,NaN,3,0.001120,False,64
17,17,COMPLETE,5.543350,4,128,0.305409,8.0,NaN,2,2,NaN,4,0.000400,False,96
47,47,COMPLETE,5.547628,4,128,0.262172,32.0,NaN,2,4,NaN,4,0.001843,False,64
49,49,COMPLETE,5.547918,4,128,0.169691,32.0,NaN,2,4,NaN,4,0.002348,False,64
45,45,COMPLETE,5.550861,4,128,0.311309,8.0,NaN,2,4,0.000013,3,0.001130,True,96
54,54,COMPLETE,5.554091,4,128,0.282723,32.0,NaN,2,4,NaN,3,0.002591,False,64


Latest checkpoint: trial-059 epoch 53


,epoch,training_batch_objective,seconds,validation.selection_loss,validation.weighted_cross_entropy,validation.location_rmse_norm
0,51,5.505903,9.633584,5.571303,1.088446,0.129074
1,52,5.504429,9.576289,5.565111,1.086788,0.131170
2,53,5.503367,9.637706,5.558775,1.085924,0.129156


## 4. Inspect selection across seeds

The top three completed trials are fitted again with three seeds and no Optuna pruning. Mean best validation objective selects the configuration. The median of its three best epochs fixes the duration of every fresh refit. Seed standard deviation measures optimisation variability, not uncertainty from different sampled matches.

In [4]:
selections = {}
for mode, title in modes.items():
    path = report / mode / "selection.json"
    print(title)
    if path.exists():
        selected = read_json(path)
        selections[mode] = selected
        display(pd.DataFrame([{k: c[k] for k in ["trial", "mean_validation_loss", "std_validation_loss", "refit_epochs"]} for c in selected["candidates"]]))
        display(pd.DataFrame([selected["winner"]["config"]]))
        display(pd.DataFrame(selected["winner"]["seeds"]).drop(columns="config"))
    else:
        print("Finalist selection has not completed.")

Seq2Event team streams
Finalist selection has not completed.
Full-quarter causal decoder


,trial,mean_validation_loss,std_validation_loss,refit_epochs
0,38,5.531943,0.004300,56
1,46,5.539129,0.002846,57
2,56,5.542971,0.001921,56


,mode,width,embedding_dim,club_embedding_dim,heads,layers,ff_multiplier,dense_dim,dropout,learning_rate,weight_decay,window
0,joint,64,8,4,4,4,2,128,0.261275,0.0019,0.0,0


,best_epoch,validation_loss,epochs_completed,stopping_reason,seed
0,50,5.531118,60,early_stopping,2026
1,57,5.536596,60,epoch_limit,2027
2,56,5.528116,60,epoch_limit,2028


## 5. Examine learning curves near the selected epoch

Check whether validation performance has plateaued or is still improving near the training limit. A winner at a search boundary or epoch limit is a reason to question whether the search was sufficient. The batch training objective and global validation objective use different reductions for coordinate RMSE, so their absolute values are not directly interchangeable.

In [5]:
for mode, selected in selections.items():
    print(modes[mode])
    trial = selected["winner"]["trial"]
    for seed_result in selected["winner"]["seeds"]:
        seed = seed_result["seed"]
        path = report / mode / f"finalist-{trial:03d}-seed-{seed}" / "history.json"
        if path.exists():
            print("Seed:", seed, "Best epoch:", seed_result["best_epoch"], "Stopped:", seed_result["stopping_reason"])
            display(pd.json_normalize(read_json(path)).tail(10))

Full-quarter causal decoder
Seed: 2026 Best epoch: 50 Stopped: early_stopping


,epoch,training_batch_objective,seconds,validation.selection_loss,validation.weighted_cross_entropy,validation.location_rmse_norm
50,51,5.528664,8.835251,5.542239,1.082584,0.129321
51,52,5.530463,8.820819,5.536840,1.081676,0.128461
52,53,5.525173,8.800582,5.532865,1.080706,0.129335
53,54,5.524923,8.856320,5.534871,1.081032,0.129710
54,55,5.522616,8.849219,5.549812,1.084179,0.128916
55,56,5.522172,8.932748,5.543988,1.083292,0.127527
56,57,5.519924,8.806782,5.537390,1.081716,0.128811
57,58,5.515928,8.850064,5.531849,1.080622,0.128740
58,59,5.516448,8.807111,5.538849,1.081717,0.130263
59,60,5.512737,8.833788,5.539289,1.082010,0.129241


Seed: 2027 Best epoch: 57 Stopped: epoch_limit


,epoch,training_batch_objective,seconds,validation.selection_loss,validation.weighted_cross_entropy,validation.location_rmse_norm
50,51,5.533191,8.792874,5.541855,1.082489,0.129410
51,52,5.530478,8.823862,5.541469,1.082502,0.128959
52,53,5.528277,8.872200,5.540063,1.082210,0.129014
53,54,5.530552,8.746426,5.547107,1.083645,0.128882
54,55,5.528593,8.845304,5.555193,1.085332,0.128535
55,56,5.525267,8.819857,5.545139,1.083107,0.129605
56,57,5.522508,8.821186,5.536596,1.081462,0.129286
57,58,5.521566,8.866404,5.540498,1.082397,0.128513
58,59,5.520563,8.829898,5.540758,1.082373,0.128893
59,60,5.519380,8.832431,5.541579,1.082617,0.128494


Seed: 2028 Best epoch: 56 Stopped: epoch_limit


,epoch,training_batch_objective,seconds,validation.selection_loss,validation.weighted_cross_entropy,validation.location_rmse_norm
50,51,5.526438,8.831157,5.533134,1.080822,0.129027
51,52,5.522928,8.853351,5.536846,1.081720,0.128248
52,53,5.520350,8.808371,5.551603,1.084536,0.128921
53,54,5.521296,8.833789,5.541367,1.082535,0.128690
54,55,5.520198,8.807934,5.536241,1.081479,0.128846
55,56,5.517240,8.867001,5.528116,1.080055,0.127840
56,57,5.514569,8.831543,5.544835,1.083168,0.128995
57,58,5.514403,8.830597,5.538375,1.082073,0.128011
58,59,5.512350,8.799427,5.541552,1.082876,0.127175
59,60,5.510991,8.896702,5.535261,1.081443,0.128044


## 6. Compare each model with its own transition baseline

The main classification task is ordinary next action for Seq2Event and joint next team–action/boundary for the decoder. Location errors are in dimensionless venue-normalised coordinates. Test metrics appear only after both studies and all final refits finish. No model or seed is chosen using these results.

In [6]:
metrics_path = report / "metrics.csv"
aggregate_path = report / "metrics-summary.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    aggregates = pd.read_csv(aggregate_path)
    for mode, title in modes.items():
        print(title)
        classification = "action" if mode == "team_stream" else "joint"
        columns = ["split", "model", "seed", "selection_loss", f"{classification}_accuracy", f"{classification}_log_loss", f"{classification}_macro_f1", "location_rmse_norm"]
        display(metrics.loc[metrics["mode"].eq(mode), columns])
        wanted = [f"{classification}_accuracy", f"{classification}_log_loss", f"{classification}_macro_f1", "location_rmse_norm"]
        display(aggregates.loc[aggregates["mode"].eq(mode) & aggregates["split"].eq("test") & aggregates["metric"].isin(wanted)])
else:
    print("Final evaluation has not completed. The current study cannot yet support a new test-performance claim.")

Final evaluation has not completed. The current study cannot yet support a new test-performance claim.


## 7. Inspect confusion matrices and rare actions

Overall accuracy can hide poor performance on rare actions. The complete confusion matrices and per-class scores are saved for every seed. This display uses the predeclared first seed, 2026, rather than selecting the most favourable test run. Decoder team and action marginal diagnostics are conditional on an ordinary-action target.

In [7]:
for mode, title in modes.items():
    print(title)
    classification = "action" if mode == "team_stream" else "joint"
    matrix = report / mode / f"test-transformer-seed-2026-{classification}-confusion.csv"
    classes = report / mode / "test-transformer-seed-2026-per-class.csv"
    if matrix.exists():
        with pd.option_context("display.max_rows", 30, "display.max_columns", 30):
            display(pd.read_csv(matrix, index_col=0))
        display(pd.read_csv(classes))
    else:
        print("Test confusion matrices are not available yet.")

Seq2Event team streams
Test confusion matrices are not available yet.
Full-quarter causal decoder
Test confusion matrices are not available yet.


## 8. Interpret the evidence

The studies use a substantially larger search than the initial four-candidate experiment, but 60 trials cannot establish a global optimum. Seq2Event's class-weighted action loss prioritises classes differently from ordinary accuracy or unweighted log loss. Its action and location performance should be assessed separately.

The 2026 benchmark has already informed earlier project discussions. It is not an untouched holdout. Quality masks favour cleaner recorded passages, and event predictions do not establish the effectiveness of a tactical intervention. The full protocol, reproduction commands and resume behaviour are documented in `docs/event-optuna.md`.